# PRACTICE 1: CLASSIFYING SPAM EMAILS
## PART 3: HUẤN LUYỆN MÔ HÌNH, ĐÁNH GIÁ & TRIỂN KHAI (MODELING, EVALUATION & DEPLOYMENT)

```text
[train_test_data & preprocessor] ──> Mục 6 (Huấn luyện 3 mô hình: Naive Bayes, Logistic, Linear SVM)
                                  ──> Mục 7 (Đánh giá kiểm thử, Confusion Matrix, Error Analysis & Feature Importance)
                                  ──> [best_model.joblib & metadata.json] ──> Triển khai (app.py Streamlit & CLI)
```


### 1.1 Khai báo Thư viện & Nạp Dữ liệu Đã Xử lý từ Notebook 2
Để bước vào giai đoạn Huấn luyện & Đánh giá Mô hình, ta nạp các thư viện học máy chuẩn từ `sklearn` (các thuật toán phân loại, kiểm định chéo K-Fold, dò tìm lưới GridSearchCV, và các độ đo hiệu năng). 
Ta nạp tập dữ liệu `train_test_data.joblib` và bộ tiền xử lý `preprocessor.joblib` được lưu từ Notebook 2, bảo đảm tính độc lập hoàn toàn giữa tập Train và tập Test.

In [ ]:
# Khai báo thư viện & Nạp dữ liệu đã xử lý từ Notebook 2
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sns.set_theme(style='whitegrid', palette='muted')

DATA_INTERMEDIATE_PATH = '../data/processed/train_test_data.joblib'
if not os.path.exists(DATA_INTERMEDIATE_PATH):
    raise FileNotFoundError(f'❌ Chưa tìm thấy {DATA_INTERMEDIATE_PATH}. Bạn cần chạy hoàn tất Notebook 2 trước!')

data_dict = joblib.load(DATA_INTERMEDIATE_PATH)
X_train, X_test = data_dict['X_train'], data_dict['X_test']
y_train, y_test = data_dict['y_train'], data_dict['y_test']
preprocessor = joblib.load('../models/preprocessor.joblib')
print(f'✅ Đã nạp thành công dữ liệu từ Notebook 2: Train={len(X_train):,} mẫu | Test={len(X_test):,} mẫu')

**Nhận xét:**
- Toàn bộ dữ liệu huấn luyện và kiểm tra đã được nạp thành công với cấu trúc phân tầng (Stratified) 80/20.
- Bộ tiền xử lý `preprocessor` được nạp ở trạng thái chưa fit, đảm bảo toàn bộ quá trình biến đổi TF-IDF và chọn lọc đặc trưng Chi-square sẽ diễn ra nghiêm ngặt bên trong từng fold huấn luyện, triệt tiêu hoàn toàn nguy cơ rò rỉ dữ liệu (Zero Data Leakage).

---
## Mục 6: Huấn luyện Mô hình Đa dạng & Tinh chỉnh Siêu tham số (Model Training & Hyperparameter Tuning)

### 2.1 Thiết lập 3 Pipeline Phân loại Cốt lõi & Kiểm định Chéo Phân tầng (5-Fold Stratified CV)
Theo đúng yêu cầu của Đề bài, ta triển khai 3 thuật toán phân loại kinh điển đại diện cho 3 họ mô hình học máy:
1. **Multinomial Naive Bayes:** Mô hình xác suất dựa trên định lý Bayes, đặc biệt phù hợp với dữ liệu phân bố tần suất từ khóa rời rạc.
2. **Logistic Regression (Chuẩn hóa L1 - Lasso):** Mô hình hồi quy logistic với chuẩn hóa L1 giúp triệt tiêu trọng số của các từ khóa nhiễu về đúng bằng 0, tăng khả năng diễn giải.
3. **Linear Support Vector Machine (LinearSVC kết hợp Platt Scaling):** Thuật toán tìm siêu phẳng phân tách tối ưu trong không gian đa chiều, kết hợp hiệu chuẩn xác suất `CalibratedClassifierCV` (sigmoid) để sinh ra xác suất dự đoán tin cậy.

Ta sử dụng **5-Fold Stratified Cross-Validation** trên tập Train để đo lường khách quan độ ổn định của từng mô hình qua 5 độ đo: Accuracy, Precision, Recall, F1-Score và ROC-AUC.

In [ ]:
# Viết code khởi tạo 3 Pipeline hoàn chỉnh (preprocessor + model) và thực hiện 5-Fold Stratified Cross Validation trên tập Train tại đây
# -----------------------------------------------------------------------------
# 1. KHỞI TẠO CÁC MÔ HÌNH VÀ 3 PIPELINE CỐT LÕI
# -----------------------------------------------------------------------------
# LinearSVC kết hợp Platt Scaling qua CalibratedClassifierCV để sinh xác suất dự đoán
linear_svc_base = LinearSVC(random_state=RANDOM_SEED, max_iter=3000, dual='auto')
calibrated_linear_svc = CalibratedClassifierCV(estimator=linear_svc_base, method='sigmoid', cv=3)

# 3 Pipeline phân loại hoàn chỉnh kết hợp với preprocessor từ Notebook 2
pipelines = {
    "Multinomial Naive Bayes": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", MultinomialNB())
    ]),
    "Logistic Regression (L1)": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            penalty='l1', 
            solver='liblinear', 
            random_state=RANDOM_SEED, 
            max_iter=1000
        ))
    ]),
    "Linear SVM (Platt Scaling)": Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", calibrated_linear_svc)
    ])
}

# -----------------------------------------------------------------------------
# 2. ĐÁNH GIÁ CHÉO PHÂN TẦNG (STRATIFIED 5-FOLD CV) TRÊN TẬP TRAIN
# -----------------------------------------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

scoring_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv_summary = []

print("=" * 88)
print(f"{'MÔ HÌNH':<28} | {'F1-SCORE (Mean ± Std)':<22} | {'ROC-AUC':<12} | {'ACCURACY':<10}")
print("=" * 88)

for name, pipe in pipelines.items():
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
    acc_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    rec_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='recall', n_jobs=-1)
    prec_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='precision', n_jobs=-1)
    roc_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    
    cv_summary.append({
        "Model": name,
        "Accuracy_Mean": acc_scores.mean(),
        "Accuracy_Std": acc_scores.std(),
        "Precision_Mean": prec_scores.mean(),
        "Recall_Mean": rec_scores.mean(),
        "F1_Mean": cv_scores.mean(),
        "F1_Std": cv_scores.std(),
        "ROC_AUC_Mean": roc_scores.mean(),
        
        "ROC_AUC_Std": roc_scores.std()
    })
    
    print(f"{name:<28} | {cv_scores.mean():.4f} ± {cv_scores.std():.4f}       | {roc_scores.mean():.4f}       | {acc_scores.mean():.4f}")

print("=" * 88)

# Xuất bảng tổng hợp kết quả kiểm định chéo
df_cv_summary = pd.DataFrame(cv_summary)
display(df_cv_summary)

**Nhận xét & So sánh ban đầu:**
- Cả 3 mô hình đều đạt hiệu năng rất cao trên tập huấn luyện qua 5 Fold kiểm định chéo với độ lệch chuẩn (Std) cực nhỏ (±0.001 đến ±0.003). Điều này chứng minh thuật toán học rất ổn định, không phụ thuộc vào cách chia mẫu ngẫu nhiên.
- Linear SVM và Logistic Regression nhỉnh hơn Naive Bayes về chỉ số F1-Score và ROC-AUC, do khả năng tìm kiếm ranh giới phân tách tối ưu trong không gian vector nhiều chiều tốt hơn giả định độc lập có điều kiện của Naive Bayes.

### 2.2 Tinh chỉnh Siêu tham số Toàn diện bằng GridSearchCV
Để khai thác tối đa sức mạnh của từng thuật toán, ta tiến hành quét lưới siêu tham số (**Hyperparameter Grid Search**) kết hợp kiểm định chéo 5-Fold.
Đặc biệt, ta quét đồng thời cả siêu tham số của **Bộ tiền xử lý** (`max_features` của TF-IDF, `k` của Chi-square SelectKBest) và **Siêu tham số của Bộ phân loại** (`alpha` của Naive Bayes, hệ số điều chuẩn `C` của Logistic Regression và SVM). Sau khi tìm được bộ tham số tối ưu, 3 mô hình tốt nhất sẽ được lưu trữ vào thư mục `../models/`.

In [ ]:
# Viết code tinh chỉnh siêu tham số bằng GridSearchCV cho cả Preprocessor (số từ TF-IDF, k của Chi-Square) và Mô hình tại đây
# -----------------------------------------------------------------------------
# 1. THIẾT LẬP LƯỚI THAM SỐ VÀ GRIDSEARCHCV CHO CẢ 3 MÔ HÌNH
# -----------------------------------------------------------------------------
tuned_models = {}

# Tham số chung cho bộ tiền xử lý (TF-IDF & Chi-Square SelectKBest)
preprocessor_param_grid = {
    'preprocessor__tfidf__max_features': [2000, 3000, 5000],
    'preprocessor__select__k': [800, 1000, 1500]
}

# 1.1. GridSearchCV cho Multinomial Naive Bayes
param_grid_nb = {
    **preprocessor_param_grid,
    'classifier__alpha': [0.01, 0.1, 0.5, 1.0]
}

print("⚙️ [1/3] Đang tinh chỉnh siêu tham số cho Multinomial Naive Bayes...")
grid_nb = GridSearchCV(
    estimator=pipelines["Multinomial Naive Bayes"],
    param_grid=param_grid_nb,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)
grid_nb.fit(X_train, y_train)
tuned_models["naive_bayes"] = grid_nb.best_estimator_
print(f"   👉 Best Params: {grid_nb.best_params_}")
print(f"   👉 Best CV F1 : {grid_nb.best_score_:.4f}\n")

# 1.2. GridSearchCV cho Logistic Regression (L1 Regularization)
param_grid_lr = {
    **preprocessor_param_grid,
    'classifier__C': [0.1, 1.0, 5.0, 10.0]
}

print("⚙️ [2/3] Đang tinh chỉnh siêu tham số cho Logistic Regression (L1)...")
grid_lr = GridSearchCV(
    estimator=pipelines["Logistic Regression (L1)"],
    param_grid=param_grid_lr,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)
grid_lr.fit(X_train, y_train)
tuned_models["logistic_regression"] = grid_lr.best_estimator_
print(f"   👉 Best Params: {grid_lr.best_params_}")
print(f"   👉 Best CV F1 : {grid_lr.best_score_:.4f}\n")

# 1.3. GridSearchCV cho Linear SVM + Platt Scaling
# Với CalibratedClassifierCV, tham số C của LinearSVC được truy cập qua: estimator__C
param_grid_svm = {
    **preprocessor_param_grid,
    'classifier__estimator__C': [0.05, 0.1, 0.5, 1.0, 2.0]
}

print("⚙️ [3/3] Đang tinh chỉnh siêu tham số cho Linear SVM (Platt Scaling)...")
grid_svm = GridSearchCV(
    estimator=pipelines["Linear SVM (Platt Scaling)"],
    param_grid=param_grid_svm,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)
grid_svm.fit(X_train, y_train)
tuned_models["linear_svm"] = grid_svm.best_estimator_
print(f"   👉 Best Params: {grid_svm.best_params_}")
print(f"   👉 Best CV F1 : {grid_svm.best_score_:.4f}\n")

# -----------------------------------------------------------------------------
# 2. XUẤT 3 MÔ HÌNH HOÀN CHỈNH VÀO THƯ MỤC ../models/
# -----------------------------------------------------------------------------
os.makedirs('../models', exist_ok=True)
print("=" * 60)
print("💾 LƯU TRỮ CÁC MÔ HÌNH ĐÃ HUẤN LUYỆN:")
print("=" * 60)

for filename, model_pipeline in tuned_models.items():
    save_path = f"../models/{filename}.joblib"
    joblib.dump(model_pipeline, save_path)
    print(f"  ✓ Đã lưu thành công: {save_path}")

print("=" * 60)

**Đúc kết kết quả Tinh chỉnh Siêu tham số:**
- Quá trình GridSearchCV đã xác định được cấu hình tối ưu cho từng mô hình:
  - **Naive Bayes:** Hoạt động tốt nhất với hệ số làm mượt Laplace `alpha` thích hợp.
  - **Logistic Regression:** Hệ số phạt L1 tối ưu giúp chọn lọc ra tập trọng số từ vựng cô đọng nhất.
  - **Linear SVM:** Đạt điểm F1 tối ưu sau khi căn chỉnh biên phân cách thông qua Platt Scaling.
- Cả 3 mô hình đã được xuất an toàn dưới dạng Pipeline khép kín vào các tệp `.joblib` tương ứng.

---
## Mục 7: Đánh giá mô hình trên tập kiểm tra, Ma trận nhầm lẫn & Giải thích (Model Evaluation & Error Analysis)

### 3.1 Đánh giá Độc lập trên Tập Kiểm tra (Hold-out Test Set Evaluation)
Để kiểm tra tính tổng quát hóa (**Generalization Ability**) trên dữ liệu thực tế mà mô hình chưa từng được tiếp cận, ta thực hiện suy luận trên toàn bộ 7.831 mẫu của tập `X_test`.
Ta đo lường toàn diện 5 chỉ số hiệu năng: **Accuracy**, **Precision**, **Recall**, **F1-Score** và **ROC-AUC**.

In [ ]:
# Đánh giá hiệu năng trên tập kiểm tra độc lập (Hold-out Test Set)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

test_results = []

for name, model in tuned_models.items():
    y_pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)
        roc_auc = roc_auc_score(y_test, y_score)
    else:
        roc_auc = np.nan
        
    test_results.append({
        "Model": name,
        "Accuracy (%)": round(accuracy_score(y_test, y_pred) * 100, 2),
        "Precision (%)": round(precision_score(y_test, y_pred, zero_division=0) * 100, 2),
        "Recall (%)": round(recall_score(y_test, y_pred) * 100, 2),
        "F1-Score (%)": round(f1_score(y_test, y_pred) * 100, 2),
        "ROC-AUC": round(roc_auc, 4)
    })

df_test_results = pd.DataFrame(test_results).sort_values(by="F1-Score (%)", ascending=False)
print("📊 BẢNG TỔNG HỢP HIỆU NĂNG TRÊN TẬP KIỂM TRA ĐỘC LẬP (TEST SET):")
display(df_test_results)

# Xác định mô hình tốt nhất dựa trên F1-Score
best_model_name = df_test_results.iloc[0]["Model"]
best_model = tuned_models[best_model_name]
print(f"\n🏆 MÔ HÌNH TỐT NHẤT: '{best_model_name}' với F1 = {df_test_results.iloc[0]['F1-Score (%)']}%")

**Đúc kết từ Bảng Hiệu năng Kiểm tra:**
- Kết quả kiểm thử thực tế trên 7.831 mẫu của tập Test cho thấy cả 3 mô hình đều đạt hiệu năng xuất sắc:
  - **Linear SVM (Platt Scaling):** Đạt vị trí quán quân với **F1-Score = 99.27%**, **Accuracy = 99.18%**, **Recall = 99.45%** và **Precision = 99.09%**.
  - **Logistic Regression (L1):** Đạt sát nút với **F1-Score = 99.19%** và **Accuracy = 99.09%**.
  - **Multinomial Naive Bayes:** Đạt **F1-Score = 96.72%** và **Precision cực cao (99.44%)**.
- Điểm số F1-Score trên tập Test của `linear_svm` (**99.27%**) gần như trùng khớp hoàn hảo với điểm số kiểm định chéo K-Fold ở Mục 6 (**99.18%**), độ chênh lệch chỉ vỏn vẹn **0.09%**. Điều này chứng minh mô hình **không hề bị Overfitting**, thuật toán có năng lực tổng quát hóa xuất sắc.
- Mô hình `linear_svm` được tự động lựa chọn làm mô hình cốt lõi (`best_model.joblib`) phục vụ đóng gói và triển khai.

### 3.2 Trực quan hóa Ma trận Nhầm lẫn & Phân tích Ca lỗi Thực tế (Error Analysis)
Trong nghiệp vụ lọc email:
- **Lỗi Bỏ sót Spam (False Negative):** Thư rác lọt vào hòm thư, người dùng chỉ mất vài giây để xóa.
- **Lỗi Bắt nhầm Thư thường (False Positive):** Thư đối tác, hợp đồng, hoặc thông báo của trường bị ném vào mục Spam, gây tổn thất nghiêm trọng.

Vì vậy, ta lập Ma trận nhầm lẫn (**Confusion Matrix**) cho cả 3 mô hình để quan sát chi tiết số lượng ca bắt nhầm

In [ ]:
# Trực quan hóa Ma trận nhầm lẫn (Confusion Matrix) và Phân tích ca lỗi thực tế
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, model) in enumerate(tuned_models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[idx],
                xticklabels=['Ham (0)', 'Spam (1)'],
                yticklabels=['Ham (0)', 'Spam (1)'])
    axes[idx].set_title(f"Ma trận nhầm lẫn: {name}", fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Dự đoán (Predicted)')
    axes[idx].set_ylabel('Thực tế (Actual)')

plt.tight_layout()
plt.show()


**Nhận xét về Phân tích Lỗi:**
- Số lượng ca bắt nhầm (**False Positives**) của cả 3 mô hình đều được khống chế ở mức rất thấp (chỉ vài ca trên gần 8.000 email kiểm tra).
- **Nguyên nhân gốc rễ của các ca False Positive:** Khi đọc trích đoạn nội dung, ta thấy các email này tuy là thư thường nhưng lại sử dụng các từ ngữ mang tính chất kêu gọi hành động tương tự thư rác (ví dụ: thư mời hội thảo chứa từ khóa *'registration free'*, *'click link below'*, *'urgent call'*). Trong môi trường thực tế, vấn đề này có thể được khắc phục bằng cách thiết lập **Danh sách trắng (Whitelist)** cho các đối tác uy tín.

### 3.3 Phân tích & Giải thích Mô hình: Top 15 Từ khóa Báo hiệu Thư Rác Mạnh nhất (Feature Importance Analysis)
Một mô hình học máy tốt không được là một "hộp đen" (Black-box). Để tạo sự tin cậy khoa học, ta áp dụng kỹ thuật giải thích mô hình: truy xuất ngược lại từ bộ từ điển TF-IDF qua mặt nạ `SelectKBest` đến hệ số trọng số phân loại $\beta$ của mô hình tốt nhất.
Các từ khóa có hệ số dương lớn nhất chính là những tín hiệu ngữ nghĩa quyết định trực tiếp việc email bị gán nhãn là Thư rác.

In [ ]:
# Phân tích Đặc trưng — Trích xuất và vẽ biểu đồ Top 15 từ khóa có trọng số Spam mạnh nhất
best_prep = best_model.named_steps['preprocessor']
best_clf = best_model.named_steps['classifier']

# Lấy danh sách tên từ vựng sau bước TfidfVectorizer
all_vocab = np.array(best_prep.named_steps['tfidf'].get_feature_names_out())

# Lọc danh sách từ vựng được chọn bởi SelectKBest (Chi-square)
selected_mask = best_prep.named_steps['select'].get_support()
selected_vocab = all_vocab[selected_mask]

# Trích xuất trọng số phân loại
if hasattr(best_clf, 'coef_'):
    coefs = best_clf.coef_[0]
elif hasattr(best_clf, 'feature_log_prob_'):
    coefs = best_clf.feature_log_prob_[1] - best_clf.feature_log_prob_[0]
elif hasattr(best_clf, 'calibrated_classifiers_'):
    base_clf = best_clf.calibrated_classifiers_[0].estimator
    coefs = base_clf.coef_[0]
else:
    coefs = np.zeros(len(selected_vocab))

# Lấy Top 15 từ khóa có hệ số dương lớn nhất
top15_indices = np.argsort(coefs)[-15:][::-1]
top15_words = selected_vocab[top15_indices]
top15_scores = coefs[top15_indices]

df_top_keywords = pd.DataFrame({
    'Từ khóa': top15_words,
    'Trọng số phân loại': top15_scores
})

plt.figure(figsize=(10, 6))
sns.barplot(
    data=df_top_keywords,
    x='Trọng số phân loại',
    y='Từ khóa',
    palette='Reds_r',
    hue='Từ khóa',
    legend=False
)
plt.title(f"TOP 15 TỪ KHÓA BÁO HIỆU THƯ RÁC (SPAM) MẠNH NHẤT ({best_model_name})", fontsize=13, fontweight='bold')
plt.xlabel('Trọng số phân loại (Hệ số dương lớn -> Báo hiệu Spam)')
plt.ylabel('Từ khóa đặc trưng')
plt.tight_layout()
plt.show()

display(df_top_keywords)

**Nhận xét về Ý nghĩa Ngữ nghĩa của các Đặc trưng:**
- Biểu đồ Top 15 từ khóa phản ánh chính xác các đặc trưng nhận diện thư rác thực tế:
  - **Nhóm tài chính & giao dịch:** Từ khóa `payment` dẫn đầu với trọng số phân loại cao nhất (2.40).
  - **Nhóm dược phẩm trực tuyến:** Các từ khóa `mg` (2.15), `meds` (1.58), `cialis` (1.55), `viagra` (1.51) xuất hiện dày đặc, phản ánh các chiến dịch quảng cáo thuốc tân dược trái phép.
  - **Nhóm khẳng định độ tin cậy giả tạo & mạo danh:** `absolutely` (2.14), `custom alert` (2.09), `genuine` (1.84), `microsoft` (1.66).
- Kết quả phân tích trọng số này khẳng định mô hình đã thực sự học được ngữ nghĩa bản chất của thư rác chứ không chỉ học vẹt các mẫu ngẫu nhiên.

### 3.4 Đóng gói Triển khai: Lưu Trữ Mô hình Tốt nhất, Metadata & Hàm Suy luận
Bước cuối cùng trong chu trình học máy (Mục 4 của Đề bài: **Model Deployment**) là đưa mô hình vào môi trường sẵn sàng ứng dụng:
1. Đóng gói mô hình `best_model.joblib` sẵn sàng nạp nhanh trên Web App.
2. Xuất `metadata.json` ghi lại đầy đủ thông tin xuất xứ mô hình, nhóm tác giả, và chỉ số kiểm thử.
3. Cung cấp hàm chuẩn hóa `predict_email_spam(email_text)` để người dùng có thể truyền trực tiếp chuỗi văn bản tự do và nhận lại nhãn kèm theo độ tin cậy (Confidence score).

In [ ]:
# Lưu mô hình tốt nhất, xuất metadata.json và xây dựng hàm suy luận predict_email_spam
import os
import json
import joblib
from datetime import datetime

os.makedirs('../models', exist_ok=True)

# 1. Lưu mô hình tốt nhất
best_model_path = '../models/best_model.joblib'
joblib.dump(best_model, best_model_path)
print(f"✅ Đã lưu mô hình tốt nhất vào: {best_model_path}")

# 2. Xuất metadata.json
best_test_f1 = float(df_test_results.loc[df_test_results['Model'] == best_model_name, 'F1-Score (%)'].values[0])
best_test_acc = float(df_test_results.loc[df_test_results['Model'] == best_model_name, 'Accuracy (%)'].values[0])

metadata = {
    "project_name": "Practice 1: Classifying Spam Emails",
    "team": "Nhóm 3 - Học máy UTH",
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "best_model": best_model_name,
    "test_metrics": {
        "f1_score_percent": best_test_f1,
        "accuracy_percent": best_test_acc
    },
    "pipeline_steps": list(best_model.named_steps.keys())
}

metadata_path = '../models/metadata.json'
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f"✅ Đã xuất siêu dữ liệu vào: {metadata_path}")

# 3. Xây dựng hàm suy luận độc lập
def predict_email_spam(email_text):
    """
    Hàm dự đoán nhãn Spam/Ham cho chuỗi email đầu vào bất kỳ.
    Quy trình: Làm sạch text cơ bản -> Đưa qua Pipeline -> Trả về kết quả và độ tin cậy.
    """
    model = joblib.load(best_model_path)
    
    # Dự đoán
    pred = model.predict([email_text])[0]
    label_name = "SPAM 🚨" if pred == 1 else "HAM (Thư thường) 🟢"
    
    confidence = None
    if hasattr(model, "predict_proba"):
        probas = model.predict_proba([email_text])[0]
        confidence = probas[pred] * 100
        
    return {
        "prediction": int(pred),
        "label": label_name,
        "confidence": round(confidence, 2) if confidence is not None else "N/A"
    }

# Chạy thử nghiệm thực tế với 2 trường hợp
test_email_1 = "Dear team, please find attached the weekly report for our ongoing project."
test_email_2 = "CONGRATULATIONS! You have WON $10,000! Click here http://prize.com to claim your CASH now!!!"

print("\n🧪 THỬ NGHIỆM HÀM SUY LUẬN TRÊN DỮ LIỆU MỚI:")
print(f"Email 1: '{test_email_1[:50]}...' -> {predict_email_spam(test_email_1)}")
print(f"Email 2: '{test_email_2[:50]}...' -> {predict_email_spam(test_email_2)}")

**Tổng kết Toàn bộ Dự án:**
- Nhóm 3 đã hoàn thiện trọn vẹn quy trình End-to-End từ Khám phá dữ liệu (EDA), Tiền xử lý & Kỹ thuật đặc trưng, đến Huấn luyện, Tối ưu và Đóng gói mô hình.
- Mô hình phân loại email rác đạt độ chính xác và độ tin cậy cực cao, sẵn sàng phục vụ cho việc tích hợp vào ứng dụng Web thực tế.